# Tata Steel Maintenance Wizard - Kaggle Qwen3-8B Backend Notebook

This notebook is the backend-only version of the Maintenance Wizard project. It reconstructs the current backend package, generates the steel maintenance data, loads Qwen3-8B on Kaggle GPU, and exposes a simple `ask()` function for testing.

Run it on Kaggle with GPU enabled. Recommended accelerator: T4 x2, P100, or better. The model is loaded in 4-bit to fit typical Kaggle GPUs.

In [ ]:
# Cell 1: Install only model/runtime extras. Do NOT reinstall numpy/pandas/sklearn on Kaggle.
# If you already ran the older notebook that installed numpy/pandas, use Kaggle: Session options -> Restart session or Factory reset, then rerun.
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "transformers>=4.51.0",
    "accelerate>=0.34.0",
    "bitsandbytes",
    "sentencepiece",
    "safetensors",
    "huggingface_hub",
])

print("Installed Qwen runtime extras without touching Kaggle numpy/pandas/sklearn.")

# Fast sanity check. If this fails with numpy.dtype size changed, restart/factory reset the Kaggle session.
try:
    import numpy as np
    import pandas as pd
    import sklearn
    print("NumPy:", np.__version__, "Pandas:", pd.__version__, "Sklearn:", sklearn.__version__)
except Exception as exc:
    raise RuntimeError("Kaggle scientific stack is ABI-broken from a previous install. Restart/factory reset the session, then rerun this updated notebook. Original error: " + str(exc))


In [ ]:
# Cell 2: Reconstruct the backend package from the embedded project payload
import base64
import io
import os
import pathlib
import shutil
import sys
import zipfile

BASE_DIR = pathlib.Path("/kaggle/working/tata_steel_agentic_ai")
if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)
BASE_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ZIP_B64_CHUNKS = [
  '...',
]

payload = base64.b64decode("".join(PROJECT_ZIP_B64_CHUNKS))
with zipfile.ZipFile(io.BytesIO(payload), "r") as zf:
    zf.extractall(BASE_DIR)

sys.path.insert(0, str(BASE_DIR))
print("Reconstructed backend at", BASE_DIR)
print("Backend files:")
for path in sorted((BASE_DIR / "backend").glob("*.py")):
    print("-", path.name)


In [ ]:
# Cell 3: Kaggle runtime configuration
import os
import torch

# Strict Qwen3-8B mode. If this crashes, Kaggle GPU memory is insufficient; use a bigger GPU or the fallback model below.
MODEL_ID = "Qwen/Qwen3-8B"
# Fallback options if Kaggle kills the kernel while loading 8B:
# MODEL_ID = "Qwen/Qwen3-4B"
# MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

os.environ["MW_USE_LLM"] = "1"
os.environ["MW_LLM_PROVIDER"] = "local_gpu"
os.environ["MW_LLM_MODEL_ID"] = MODEL_ID
os.environ["MW_LLM_LAZY_LOAD"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} | free {free/1024**3:.2f} GiB / total {total/1024**3:.2f} GiB")
else:
    print("WARNING: GPU is not enabled. Turn on GPU in Kaggle notebook settings before loading Qwen3-8B.")


In [ ]:
# Cell 4: Patch the backend LLM wrapper to load Qwen3-8B on Kaggle GPU
# This keeps all existing agent functions, tools, memory, RAG, ML, and verifier logic.
import gc
import os
import re
import torch

from backend.llm import LocalLLM

MAX_INPUT_TOKENS = 6144
DEFAULT_MAX_NEW_TOKENS = 700


def _supports_bfloat16():
    if not torch.cuda.is_available():
        return False
    major, _minor = torch.cuda.get_device_capability(0)
    return major >= 8


def _gpu_memory_map(reserve_gib: float = 1.6):
    memory = {}
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        free_gib = free / 1024**3
        usable = max(1, int(free_gib - reserve_gib))
        memory[i] = f"{usable}GiB"
    memory["cpu"] = "24GiB"
    return memory


def _print_gpu_state(label="GPU state"):
    print(label)
    if not torch.cuda.is_available():
        print("- CUDA is not available")
        return
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"- GPU {i}: {torch.cuda.get_device_name(i)} | free {free/1024**3:.2f} GiB / total {total/1024**3:.2f} GiB")


def kaggle_qwen_load(self):
    if not self.enabled:
        return self
    try:
        if not torch.cuda.is_available():
            raise RuntimeError("GPU is not enabled. Enable GPU in Kaggle before loading Qwen3-8B.")

        gc.collect()
        torch.cuda.empty_cache()
        _print_gpu_state("Before Qwen load")

        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from huggingface_hub import login

        hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
        if hf_token:
            login(token=hf_token, add_to_git_credential=False)

        compute_dtype = torch.bfloat16 if _supports_bfloat16() else torch.float16
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            use_fast=True,
        )
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        max_memory = _gpu_memory_map()
        print("Using max_memory:", max_memory)

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            trust_remote_code=True,
            device_map="auto",
            max_memory=max_memory,
            quantization_config=bnb_config,
            torch_dtype=compute_dtype,
            low_cpu_mem_usage=True,
            use_safetensors=True,
            attn_implementation="eager",
        )
        self.model.eval()
        self.is_encoder_decoder = False
        self.load_error = ""
        _print_gpu_state("After Qwen load")
    except Exception as exc:
        self.load_error = str(exc)
        self.tokenizer = None
        self.model = None
        print("Qwen load failed:", self.load_error)
    return self


def _strip_thinking(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", str(text), flags=re.IGNORECASE | re.DOTALL).strip()
    text = text.replace("</think>", "").strip()
    return text


def kaggle_qwen_generate(self, prompt: str, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS, max_time: float = 60.0, min_new_tokens: int = 0) -> str:
    if not self.ensure_available():
        return ""
    try:
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a serious agentic AI maintenance copilot for steel manufacturing. "
                    "Use the provided tool facts, avoid hallucination, write direct engineer-facing answers, "
                    "and do not reveal chain-of-thought or hidden implementation details."
                ),
            },
            {"role": "user", "content": prompt + "\n\n/no_think"},
        ]
        try:
            rendered = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            rendered = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = self.tokenizer(rendered, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        return _strip_thinking(self.tokenizer.decode(new_tokens, skip_special_tokens=True))
    except Exception as exc:
        self.load_error = str(exc)
        print("Qwen generation failed:", self.load_error)
        return ""


LocalLLM.load = kaggle_qwen_load
LocalLLM.generate = kaggle_qwen_generate
print("LocalLLM patched for Kaggle Qwen local-GPU loading.")

In [ ]:
# Cell 5: Generate data, train ML models, build RAG index, and initialize the agent
import json
import pandas as pd
from pathlib import Path

from backend.data_setup import prepare_data
from backend.agent import MaintenanceWizard

summary = prepare_data(force=True)
print("Data summary:")
print(json.dumps(summary, indent=2))

wizard = MaintenanceWizard()
wizard.ensure_ready()
print("Agent ready.")
print("Asset health summary:")
display(wizard.asset_health_table().sort_values("hybrid_health_score", ascending=False).head(10))

In [ ]:
# Cell 5B: Backend intelligence upgrades - multi-source scoring, evidence confidence, procurement risk, and SFT data
import json
import pandas as pd
from pathlib import Path

from backend.decision_intelligence import build_decision_intelligence_table, top_original_vs_dynamic
from backend.finetune_data import build_qwen_sft_examples

# This table is the stronger factual packet for Qwen: original assets + active/inactive dynamic assets,
# RUL, risk, delay cost, procurement risk, evidence confidence, missing evidence, and remembered rules.
decision_table = build_decision_intelligence_table(force=True)
print("Decision-intelligence table saved to /kaggle/working/tata_steel_agentic_ai/data/asset_decision_intelligence.csv")
print("Rows:", len(decision_table))
display(decision_table[[
    "asset_id", "asset_type", "area", "active", "is_dynamic", "decision_score", "priority", "risk_band",
    "estimated_rul_days", "evidence_confidence", "procurement_risk", "delay_cost_impact_inr", "applied_rule_count"
]].head(20))

comparison_packet = top_original_vs_dynamic()
print("Top original vs dynamic comparison packet:")
print(json.dumps(comparison_packet, indent=2, default=str)[:4000])

sft_path = "/kaggle/working/qwen3_8b_maintenance_sft.jsonl"
sft_examples = build_qwen_sft_examples(sft_path, min_examples=180)
print("Generated SFT examples:", len(sft_examples))
print("Saved:", sft_path)
print("Sample SFT record:")
print(json.dumps(sft_examples[0], indent=2, ensure_ascii=False)[:3000])

# Attach for later notebook cells/debugging.
wizard.decision_intelligence_table = decision_table
wizard.top_original_vs_dynamic_packet = comparison_packet


In [ ]:
# Cell 6: Load Qwen3-8B safely
# If the kernel restarts during this cell, Kaggle killed it for OOM. Use T4x2/P100/A100 if available, or switch MODEL_ID in Cell 3 to Qwen/Qwen3-4B.
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
else:
    raise RuntimeError("GPU is not enabled. Enable GPU in Kaggle settings before running Cell 6.")

# Force local-GPU mode so the notebook does not silently use an API model.
wizard.llm.model_id = MODEL_ID
wizard.llm.provider = "local_gpu"
wizard.llm.qwen_api_base = ""
wizard.llm.qwen_api_key = ""
wizard.llm.qwen_api_model = ""

wizard.llm.load()
print("Qwen available:", wizard.llm.available)
print("Active model_id:", wizard.llm.model_id)
print("Provider:", wizard.llm.provider)
print("Remote API available:", wizard.llm.remote_available)
print("Load error:", wizard.llm.load_error)
assert wizard.llm.available, "Qwen did not load. If the kernel did not restart, read Load error above. If it restarted, it is OOM; use bigger GPU or Qwen3-4B."
assert wizard.llm.model_id == MODEL_ID == "Qwen/Qwen3-8B", "This notebook is not using Qwen3-8B. Check MODEL_ID in Cell 3."
assert wizard.llm.provider == "local_gpu" and not wizard.llm.remote_available, "Remote/API model is active; Qwen3-8B local GPU is not the active LLM."
print("Confirmed: planner, explanations, and final synthesis will use local", MODEL_ID, "through wizard.llm.generate().")


In [ ]:
# Cell 7: Helper function for notebook chat testing
from IPython.display import Markdown, display


def active_llm_status():
    return {
        "model_id": wizard.llm.model_id,
        "provider": wizard.llm.provider,
        "remote_available": bool(wizard.llm.remote_available),
        "loaded": bool(wizard.llm.available),
    }


def ask(prompt: str, show_tools: bool = False):
    print("USER:", prompt)
    print("ACTIVE LLM:", active_llm_status())
    result = wizard.chat(prompt, user_id="kaggle_maintenance_engineer")
    answer = result.get("answer") or result.get("final_answer") or ""
    display(Markdown(answer))
    print("\nLLM validation:", result.get("llm_validation"))
    synth = result.get("llm_synthesizer", {})
    print("Synthesizer:", synth.get("status"), "| active model_id:", synth.get("model_id"), "| remote API model:", synth.get("qwen_api_model"))
    if show_tools:
        print("\nTool calls:")
        for call in result.get("tool_calls", []):
            print("-", call.get("agent"), "|", call.get("tool"), "|", call.get("status"), "|", call.get("output"))
        print("\nVerifier checks:")
        for check in result.get("verifier_checks", []):
            print("-", check.get("check"), "|", check.get("status"), "|", check.get("detail"))
    return result


In [ ]:
# Cell 8B: Backend upgrade validation prompts
# These are fast checks for routing, active/dynamic ranking, evidence confidence, and final synthesis quality.
UPGRADE_TEST_PROMPTS = [
    "Compare the highest-risk original demo asset with the highest-risk dynamic asset. Show side-by-side scores, RUL, priority, applied rules, evidence confidence, and final winner.",
    "Rank active dynamic assets only. Do not include GBX-17, MTR-204, HPP-12, or PMP-09. Show priority, risk, RUL, applied rules, and evidence confidence.",
    "Using original demo assets, active dynamic assets, remembered safety rules, evidence confidence, RUL, delay impact, and procurement risk, choose exactly one asset for immediate maintenance today.",
    "What evidence is missing for a newly added asset, and how should the agent avoid inventing maintenance history?",
]

upgrade_results = []
for prompt in UPGRADE_TEST_PROMPTS:
    result = ask(prompt, show_tools=True)
    upgrade_results.append({
        "prompt": prompt,
        "answer": result.get("answer", result.get("final_answer", "")),
        "mode": result.get("mode"),
        "intent": result.get("intent"),
        "asset_id": result.get("asset_id"),
        "llm_validation": result.get("llm_validation"),
        "synthesizer_status": (result.get("llm_synthesizer") or {}).get("status"),
    })
    print("=" * 120)

pd.DataFrame(upgrade_results).to_csv("/kaggle/working/backend_upgrade_validation_outputs.csv", index=False)
print("Saved /kaggle/working/backend_upgrade_validation_outputs.csv")


In [ ]:
# Cell 8: Core judge-style smoke tests
TEST_PROMPTS = [
    "If I can maintain only one asset today, which one should I choose and why?",
    "What spare parts will I need if I replace the ladle car wheel assembly this weekend? What's the procurement lead time?",
    "Our conveyor belt on line 3 just stopped. Walk me through the first checks I should do right now.",
    "Generate a digital logbook entry for today's planned maintenance on the EAF transformer cooling system. Technician: R. Kumar. Work done: oil top-up and fan belt inspection.",
    "Lube oil temperature on the BOF tilting drive is trending: 52 C one week ago, 58 C yesterday, and 63 C today. Predict remaining useful life and recommend when to intervene.",
]

results = []
for prompt in TEST_PROMPTS:
    result = ask(prompt)
    results.append({
        "prompt": prompt,
        "answer": result.get("answer", ""),
        "mode": result.get("mode"),
        "intent": result.get("intent"),
        "asset_id": result.get("asset_id"),
        "llm_validation": result.get("llm_validation"),
        "synthesizer_status": (result.get("llm_synthesizer") or {}).get("status"),
    })
    print("=" * 120)

pd.DataFrame(results).to_csv("/kaggle/working/qwen3_8b_agent_test_outputs.csv", index=False)
print("Saved /kaggle/working/qwen3_8b_agent_test_outputs.csv")

In [ ]:
# Cell 9: Dynamic memory and rule stress tests
ask("Add a new asset: ID BFB-21, asset type Blast Furnace Blower, area Blast Furnace, criticality Critical, vibration 7.4 mm/s, temperature 86 C, current 91 A, pressure 8.1 bar, alarm count 3.", show_tools=True)
ask("Remember this safety rule: any blast furnace blower with vibration above 6.5 mm/s and temperature above 80 C must be P1 because blower failure can affect furnace stability.", show_tools=True)
ask("Rank active dynamic assets only. Do not include GBX-17, MTR-204, HPP-12, or PMP-09. Show priority, risk, RUL, applied rules, and evidence confidence.", show_tools=True)
ask("Using original demo assets, active dynamic assets, remembered safety rules, evidence confidence, RUL, delay impact, and procurement risk, choose exactly one asset for immediate maintenance today.", show_tools=True)

In [ ]:
# Cell 10: Export notebook artifacts for submission/debugging
import shutil
from pathlib import Path

artifact_dir = Path("/kaggle/working/tata_steel_agentic_ai_artifacts")
artifact_dir.mkdir(exist_ok=True)

for name in [
    "qwen3_8b_agent_test_outputs.csv",
    "backend_upgrade_validation_outputs.csv",
    "qwen3_8b_maintenance_sft.jsonl",
    "tata_steel_agentic_ai/DATA_SOURCES.md",
]:
    src = Path("/kaggle/working") / name
    if src.exists():
        shutil.copy(src, artifact_dir / src.name)

# Copy generated logs/data summaries if present.
for src in (Path("/kaggle/working/tata_steel_agentic_ai/data")).glob("*.csv"):
    if src.name in {"asset_health_summary.csv", "asset_decision_intelligence.csv", "digital_logbook.csv", "dynamic_assets.csv", "dynamic_asset_history.csv", "dynamic_rules.csv", "feedback_log.csv", "qwen3_8b_maintenance_sft.jsonl"}:
        shutil.copy(src, artifact_dir / src.name)

zip_path = shutil.make_archive(str(artifact_dir), "zip", artifact_dir)
print("Saved artifact zip:", zip_path)
